# BB84 / Ekert91 Plain Demonstration — No Attacker

This notebook demonstrates the **normal operation of the BB84 quantum key distribution protocol** without an attacker.

The assignment asks for:
- a single sequential program,
- clear sections showing which parts belong to Alice and Bob,
- random choices generated using quantum measurement instead of Python's standard random number generator,
- a final shared key,
- an error-rate check showing that no attacker is detected.

Although this file is named `Ekert91-Plain.ipynb`, the implementation below follows the **BB84 protocol** required in the assignment brief.


In [1]:
# Install the required packages.
# Run this cell first if Qiskit is not already installed in your environment.

%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 57.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=0c586cb85d676f88e0131037df159a0d1ddf9d7ecf1595d8f5c2c06eb87dce20
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# Imports used for the BB84 simulation.
# Important: We do NOT import or use Python's random module.
# All random bits and random bases are generated by measuring a quantum superposition.

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from IPython.display import display, Markdown

simulator = AerSimulator()


## Helper functions

The function below generates randomness by preparing the quantum state:

\[
\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)
\]

and then measuring it. This gives a random `0` or `1`.

This satisfies the assignment requirement: **do not use a standard Python random number generator**.


In [3]:
def quantum_random_bit():
    """
    Generate one random classical bit using quantum measurement.

    A Hadamard gate creates an equal superposition.
    Measuring the qubit gives 0 or 1 with approximately equal probability.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    result = simulator.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])


def quantum_random_bits(length):
    """Generate a list of random bits using quantum measurement only."""
    return [quantum_random_bit() for _ in range(length)]


def basis_name(basis):
    """Convert basis value into a readable label."""
    return "Z/+" if basis == 0 else "X/x"


def bit_string(bits):
    """Convert a list of bits into a compact string."""
    return "".join(str(bit) for bit in bits)


## Step 1 — Alice generates random bits and bases

Alice needs:
- random data bits,
- random encoding bases.

Basis meaning:
- `0` = Z / computational / rectilinear basis
- `1` = X / Hadamard / diagonal basis


In [4]:
# Number of qubits to send.
# You can increase this value if you want a longer final key.
N = 40

# Alice's private random choices.
alice_bits = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print("Alice bits:  ", bit_string(alice_bits))
print("Alice bases: ", [basis_name(b) for b in alice_bases])


Alice bits:   1101010110010011110001010101100100101101
Alice bases:  ['Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+']


## Step 2 — Alice prepares qubits

Alice encodes each bit using her chosen basis.

Encoding rule:
- Bit `0`, Z basis → \(|0\rangle\)
- Bit `1`, Z basis → \(|1\rangle\)
- Bit `0`, X basis → \(|+\rangle\)
- Bit `1`, X basis → \(|-\rangle\)

For the simulation, we store the bit and basis as the prepared quantum state description. The measurement behaviour is then simulated according to BB84 rules.


In [5]:
# Alice's prepared qubits are represented as (bit, basis).
# This makes it clear what Alice sends through the quantum channel.

prepared_qubits = list(zip(alice_bits, alice_bases))

print("First 10 prepared qubits as (bit, basis):")
for i, (bit, basis) in enumerate(prepared_qubits[:10]):
    print(f"Qubit {i:02d}: bit={bit}, basis={basis_name(basis)}")


First 10 prepared qubits as (bit, basis):
Qubit 00: bit=1, basis=Z/+
Qubit 01: bit=1, basis=Z/+
Qubit 02: bit=0, basis=Z/+
Qubit 03: bit=1, basis=Z/+
Qubit 04: bit=0, basis=X/x
Qubit 05: bit=1, basis=Z/+
Qubit 06: bit=0, basis=X/x
Qubit 07: bit=1, basis=Z/+
Qubit 08: bit=1, basis=X/x
Qubit 09: bit=0, basis=X/x


## Step 3 — Bob generates random measurement bases

Bob independently chooses a random basis for every qubit using quantum-generated randomness.


In [6]:
bob_bases = quantum_random_bits(N)

print("Bob bases:")
print([basis_name(b) for b in bob_bases])


Bob bases:
['Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+']


## Step 4 — Bob measures Alice's qubits

BB84 measurement rule:
- If Bob uses the same basis as Alice, he gets Alice's bit correctly.
- If Bob uses a different basis, the result is random.

The random result in the wrong-basis case is also generated using quantum measurement.


In [7]:
bob_results = []

for i in range(N):
    alice_bit = alice_bits[i]
    alice_basis = alice_bases[i]
    bob_basis = bob_bases[i]

    if bob_basis == alice_basis:
        # Correct basis: Bob obtains Alice's bit.
        measured_bit = alice_bit
    else:
        # Wrong basis: measurement outcome is random.
        measured_bit = quantum_random_bit()

    bob_results.append(measured_bit)

print("Bob results:")
print(bit_string(bob_results))


Bob results:
1011110111110011110110011100101100101101


## Step 5 — Public basis comparison and key sifting

Alice and Bob publicly compare only their bases, **not the bit values**.

They keep only the positions where their bases match.


In [8]:
matching_positions = [
    i for i in range(N)
    if alice_bases[i] == bob_bases[i]
]

alice_sifted_key = [alice_bits[i] for i in matching_positions]
bob_sifted_key = [bob_results[i] for i in matching_positions]

print("Matching positions:", matching_positions)
print("Alice sifted key:", bit_string(alice_sifted_key))
print("Bob sifted key:  ", bit_string(bob_sifted_key))
print("Sifted key length:", len(alice_sifted_key))


Matching positions: [0, 11, 12, 14, 17, 18, 22, 23, 28, 32, 35, 36, 39]
Alice sifted key: 1101100110011
Bob sifted key:   1101100110011
Sifted key length: 13


## Step 6 — Error-rate test

In a real BB84 protocol, Alice and Bob reveal a small sample of their sifted key to estimate the error rate.

For this simulation, we compare the full sifted key so the demonstration is easy to verify. With no attacker, the error rate should normally be `0.0`.


In [9]:
errors = sum(
    1 for a, b in zip(alice_sifted_key, bob_sifted_key)
    if a != b
)

error_rate = errors / len(alice_sifted_key) if alice_sifted_key else 0

threshold = 0.20

print("Errors:", errors)
print("Error rate:", round(error_rate, 3))
print("Detection threshold:", threshold)

if error_rate > threshold:
    print("Attack detected!")
else:
    print("No attack detected. The BB84 key exchange was successful.")


Errors: 0
Error rate: 0.0
Detection threshold: 0.2
No attack detected. The BB84 key exchange was successful.


## Final conclusion

The plain BB84 run shows that Alice and Bob can establish a shared sifted key when there is no attacker.  
Because Bob's measurements match Alice's bits at the positions where their bases agree, the error rate remains below the detection threshold.
